# Study 808 — Continuing Overreaction 🔁

**Does a name on a persistent recent up-streak keep running?**

Byun, Lim & Yun (2016) build a **weighted signed-momentum** score — a recency-weighted
sum of the *signs* of a stock's recent monthly returns (recent months count most). A
high positive score marks a consistent up-streak ("continuing overreaction"), which is
supposed to predict the cross-section *positively*: the streak keeps going. We take the
self-contained monthly version on a liquid US cross-section (2010-01-04 → 2026-06-30,
50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

Take a stock's last 12 monthly returns, drop the most recent month, and just look at the **signs**: up, up, down, up… Weight the *recent* months more and add them up (normalised) to get a score between −1 and +1. A score near +1 is a consistent recent up-streak. The behavioural story ('continuing overreaction') says investors keep chasing that streak, so it should keep running — buy the high-CO names, sell the low-CO ones.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=15.46, t_nw=0.58, hi_bps=152.94, lo_bps=137.48, gross_sharpe=0.14)
print('long high-CO / short low-CO spread: %+.2f bps/month (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  high-CO book %+.2f bps vs low-CO book %+.2f bps'
      % (R['hi_bps'], R['lo_bps']))
print('  gross spread Sharpe (before cost, ann.): %.2f' % R['gross_sharpe'])

long high-CO / short low-CO spread: +15.46 bps/month (NW t = +0.58)
  high-CO book +152.94 bps vs low-CO book +137.48 bps
  gross spread Sharpe (before cost, ann.): 0.14


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: a persistent monthly trend state drives both past signs and the forward month) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, signs are coin-flips). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from continuing_overreaction import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=808, n_assets=40, n_days=1800))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.02, seed=808, n_assets=40, n_days=1800))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -0.25  (should be ~0)
planted world: spread NW t = +8.61  (should light up)


## 3. The honest verdict — the famous edge does *not* replicate here

On this liquid mega-cap tape the long-high-CO / short-low-CO spread is **+15.46 bps/month** with NW *t* = **+0.58** — the *right sign* (a whisper of continuation) but **statistically indistinguishable from zero**: a monthly Sharpe of 0.14, the high-CO and low-CO books within a rounding error (+153 vs +137 bps), and only ≈+0.68σ into a 1,000-permutation placebo (p = 0.261). The seeded synthetic control recovers a *planted* continuation cleanly, so this is a genuine null on the mega-cap universe, not a bug — the continuing-overreaction premium simply does not bite on 50 well-arbitraged mega-caps. **Signal: None** (the claimed edge is absent), **Tradability: Mirage**.